# SHine-K — URFD 낙상 감지 재현 (최소 구성, Colab)

**런타임 → 모두 실행(Run all)** 만 하면 됩니다. Drive 마운트 불필요 · 결과 zip 자동 다운로드.

- 기본값 = **전체 70시퀀스**(낙상 30 + ADL 40) · 논문과 동일한 배포 임계값(sens=1.0, 재튜닝 없음)
- 빠른 동작 확인만 원하면 아래 셀에서 `N_FALL, N_ADL = 8, 8` (논문 부분집합 구성)로 바꾸세요.
- 소요: CPU 런타임 약 1–2시간 · **T4 GPU 런타임 약 20–40분** (런타임 → 런타임 유형 변경)
- 산출물: `eval_metrics.json` · `eval_per_sequence.csv` · `eval_confusion_matrix.png` · `run_manifest.json`
- 평가 로직은 `SHine-K_reproducibility_colab.ipynb` 셀과 동일(verbatim)입니다 — 정직성 원칙: 나온 수치를 그대로 보고합니다.


In [ ]:
# 1) 의존성 — Colab에는 대부분 사전 설치되어 있어 보통 수 초면 끝납니다
!pip -q install tensorflow tensorflow-hub opencv-python-headless matplotlib
print("deps ready")

In [ ]:
# 2) 전체 평가 — 설정 → 모델 로드 → 70시퀀스 평가 → 지표/그림/매니페스트 → zip 다운로드
N_FALL, N_ADL, FRAME_STRIDE = 30, 40, 1     # 빠른 확인: 8, 8, 1

import os, sys, csv, json, math, time, zipfile, hashlib, shutil, platform, urllib.request
from pathlib import Path
from datetime import datetime, timezone, timedelta
import numpy as np

t0 = time.time()
KST = timezone(timedelta(hours=9))
RUN_ID = "run_" + datetime.now(KST).strftime("%Y%m%d_%H%M%S") + "_full%d" % (N_FALL + N_ADL)
BASE_OUTPUT = Path("/content/output") if os.path.isdir("/content") else Path("./output")
RUN_DIR = BASE_OUTPUT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR)

# ── 이하 평가 로직은 재현 노트북 셀 verbatim (env 변수 대신 위 상수 사용) ──
os.environ["URFD_N_FALL"], os.environ["URFD_N_ADL"], os.environ["URFD_STRIDE"] = str(N_FALL), str(N_ADL), str(FRAME_STRIDE)

import tensorflow as tf, tensorflow_hub as hub
import cv2, io

N_FALL = int(os.environ.get("URFD_N_FALL", "8"))   # full dataset = 30
N_ADL  = int(os.environ.get("URFD_N_ADL",  "8"))   # full dataset = 40
FRAME_STRIDE = int(os.environ.get("URFD_STRIDE", "1"))
BASE_URL = "https://fenix.ur.edu.pl/~mkepski/ds/data"  # https: server 301-redirects http
CACHE = BASE_OUTPUT / "_urfd_cache"; CACHE.mkdir(parents=True, exist_ok=True)

print("Loading MoveNet MultiPose Lightning (matches the deployed model) ...")
_net = hub.load("https://tfhub.dev/google/movenet/multipose/lightning/1")
_infer = _net.signatures["serving_default"]

def movenet_kps(frame_rgb):
    # -> (kps[17x(x_px,y_px,score)], W, H) for the top-scoring person, or None
    H, W = frame_rgb.shape[:2]
    img = tf.image.resize_with_pad(tf.expand_dims(frame_rgb, 0), 256, 256)
    img = tf.cast(img, dtype=tf.int32)
    res = _infer(tf.constant(img))["output_0"].numpy()[0]   # (6,56)
    best, bs = None, -1.0
    for person in res:
        kp = person[:51].reshape(17, 3)                      # [y,x,score] in padded 256 space
        sc = float(kp[:, 2].mean())
        if sc > bs: bs, best = sc, kp
    if best is None: return None
    scale = 256.0 / max(H, W); padx = (256 - W*scale)/2; pady = (256 - H*scale)/2
    out = np.zeros((17, 3))
    for i in range(17):
        yn, xn, s = best[i]
        out[i] = [(xn*256 - padx)/scale, (yn*256 - pady)/scale, s]
    return out, W, H

# ---- faithful port of analyzePerson() from worksite_multi.html (deployed defaults, sens=1.0) ----
ACT_IDX = [0, 5, 6, 9, 10, 11, 12, 15, 16]
class FallSM:
    def __init__(self, fps=30.0, sens=1.0):
        self.fps=float(fps); self.sens=float(sens); self.reset()
    def reset(self):
        self.t=0.0; self.frame=-1; self.downSince=None; self.staticSince=None
        self.hipHist=[]; self.lastKp=None; self.state="normal"
        self.triggered=False; self.trigger_frame=None
    def _v(self, kps, i):
        x,y,s = kps[i]; return (x,y) if s>0.3 else None
    def step(self, kps, W, H):
        self.frame += 1; self.t += 1.0/self.fps; now = self.t*1000.0
        shL=self._v(kps,5); shR=self._v(kps,6); hipL=self._v(kps,11); hipR=self._v(kps,12)
        if shL is None or shR is None:
            self.state="normal"; self.lastKp=kps; return self.state
        shX=(shL[0]+shR[0])/2; shY=(shL[1]+shR[1])/2
        if hipL and hipR: hpX=(hipL[0]+hipR[0])/2; hpY=(hipL[1]+hipR[1])/2
        else:
            sw=math.hypot(shR[0]-shL[0], shR[1]-shL[1]); hpX=shX; hpY=shY+sw*1.4
        dx=shX-hpX; dy=shY-hpY
        tilt=math.degrees(math.atan2(abs(dx), abs(dy)))
        xs=[]; ys=[]
        for j in range(17):
            if kps[j][2]>0.3: xs.append(kps[j][0]); ys.append(kps[j][1])
        aspect=((max(xs)-min(xs))/max(1.0,(max(ys)-min(ys)))) if len(xs)>=2 else 0.0
        self.hipHist.append((hpY/H, self.t))
        if len(self.hipHist)>8: self.hipHist.pop(0)
        vel=0.0
        if len(self.hipHist)>=2:
            a=self.hipHist[0]; z=self.hipHist[-1]; vel=(z[0]-a[0])/max(0.001,(z[1]-a[1]))
        mv=0.0
        if self.lastKp is not None:
            sm=0.0; nm=0
            for i in ACT_IDX:
                c=kps[i]; l=self.lastKp[i]
                if c[2]>0.3 and l[2]>0.3:
                    sm+=math.hypot((c[0]-l[0])/W, (c[1]-l[1])/H); nm+=1
            mv=(sm/nm) if nm else 0.0
        self.lastKp=kps
        TILT_DOWN=52*self.sens; ASP_DOWN=1.0*self.sens; VEL_DROP=0.9/self.sens
        isDown=(tilt>TILT_DOWN) or (aspect>ASP_DOWN); rapid=vel>VEL_DROP
        if isDown:
            if not self.downSince: self.downSince=now
            self.state="fall" if (now-self.downSince>700) else "warn"
        elif rapid:
            self.state="warn"; self.downSince=None
        else:
            self.state="normal"; self.downSince=None
        if self.state=="normal":
            if mv<0.0045:
                if not self.staticSince: self.staticSince=now
                ss=(now-self.staticSince)/1000.0
                if mv<0.0016 and ss>12: self.state="inactive"
                elif ss>25: self.state="sedentary"
            else: self.staticSince=None
        else: self.staticSince=None
        if self.state in ("fall","inactive") and not self.triggered:
            self.triggered=True; self.trigger_frame=self.frame
        return self.state

def load_frames(name):
    url=f"{BASE_URL}/{name}-cam0-rgb.zip"; zp=CACHE/f"{name}.zip"
    if not zp.exists(): urllib.request.urlretrieve(url, zp)
    frames=[]
    with zipfile.ZipFile(zp) as z:
        ns=sorted(n for n in z.namelist() if n.lower().endswith((".png",".jpg",".jpeg")))
        for k,n in enumerate(ns):
            if k % FRAME_STRIDE: continue
            arr=cv2.imdecode(np.frombuffer(z.read(n), np.uint8), cv2.IMREAD_COLOR)
            if arr is not None: frames.append(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))
    return frames

seqs=[(f"fall-{i:02d}",1) for i in range(1,N_FALL+1)] + [(f"adl-{i:02d}",0) for i in range(1,N_ADL+1)]
rows=[]; TP=FP=FN=TN=0; lat=[]
for name,gt in seqs:
    try:
        frames=load_frames(name)
    except Exception as e:
        print("  skip", name, "-", e); continue
    if not frames: print("  no frames", name); continue
    sm=FallSM(fps=30.0/FRAME_STRIDE)
    for f in frames:
        r=movenet_kps(f)
        if r is None: sm.step([(0,0,0.0)]*17, f.shape[1], f.shape[0])
        else: sm.step(r[0], r[1], r[2])
    pred=1 if sm.triggered else 0
    rows.append({"seq":name,"gt":gt,"pred":pred,"trigger_frame":sm.trigger_frame,"frames":len(frames)})
    if   gt==1 and pred==1: TP+=1; lat.append(sm.trigger_frame if sm.trigger_frame is not None else len(frames))
    elif gt==0 and pred==1: FP+=1
    elif gt==1 and pred==0: FN+=1
    else: TN+=1
    print(f"  {name}: gt={gt} pred={pred} trig={sm.trigger_frame}")

prec=TP/(TP+FP) if (TP+FP) else 0.0
rec =TP/(TP+FN) if (TP+FN) else 0.0
f1  =2*prec*rec/(prec+rec) if (prec+rec) else 0.0
acc =(TP+TN)/max(1,(TP+FP+FN+TN))
EVAL={"dataset":"UR Fall Detection (URFD), cam0 RGB",
      "model":"MoveNet MultiPose Lightning (TF-Hub)",
      "state_machine":"faithful port of worksite_multi.html analyzePerson(), deployed defaults (sens=1.0)",
      "n_fall":N_FALL,"n_adl":N_ADL,"frame_stride":FRAME_STRIDE,
      "counts":{"TP":TP,"FP":FP,"FN":FN,"TN":TN},
      "precision":round(prec,4),"recall":round(rec,4),"f1":round(f1,4),"accuracy":round(acc,4),
      "detection_latency_frames":{"mean":(float(np.mean(lat)) if lat else None),
                                   "median":(float(np.median(lat)) if lat else None)},
      "caveat":"URFD = clean single-subject lab footage, fixed camera. Bounds generalization; != SME shop-floor. Evaluates pose->state-machine logic, not multi-person association. Thresholds unchanged from deployment."}
(RUN_DIR/"eval_metrics.json").write_text(json.dumps(EVAL, indent=2, ensure_ascii=False))
with open(RUN_DIR/"eval_per_sequence.csv","w",newline="") as fh:
    w=csv.DictWriter(fh, fieldnames=["seq","gt","pred","trigger_frame","frames"]); w.writeheader(); w.writerows(rows)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
cm=np.array([[TN,FP],[FN,TP]]); mx=max(1,cm.max())
fig,ax=plt.subplots(figsize=(3.5,3.3))  # publication B/W style (matches paper Fig.)
for (r_,c_),v in np.ndenumerate(cm):
    sh=0.93-0.30*(v/mx)
    ax.add_patch(Rectangle((c_,1-r_),1,1,fc=(sh,sh,sh),ec="#1a1a1a",lw=1.2))
    ax.text(c_+0.5,1-r_+0.5,int(v),ha="center",va="center",fontsize=16,fontweight="bold",color="#000")
ax.set_xlim(0,2); ax.set_ylim(0,2); ax.set_aspect("equal")
ax.set_xticks([0.5,1.5]); ax.set_xticklabels(["pred ADL","pred Fall"],fontsize=9)
ax.set_yticks([1.5,0.5]); ax.set_yticklabels(["true ADL","true Fall"],fontsize=9)
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_title("URFD fall detection (%d fall + %d ADL)\nPrecision %.2f  Recall %.2f  F1 %.2f"%(N_FALL,N_ADL,prec,rec,f1),fontsize=9.5,pad=8)
fig.savefig(RUN_DIR/"eval_confusion_matrix.png", dpi=300, bbox_inches="tight", pad_inches=0.08); plt.close(fig)
print("\nEVAL:", json.dumps(EVAL, ensure_ascii=False, indent=2))

# ── 매니페스트 + zip (재현 노트북과 동일 취지의 경량판) ──
def _sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(65536), b""): h.update(ch)
    return h.hexdigest()
manifest = {
  "run_id": RUN_ID,
  "generated_kst": datetime.now(KST).isoformat(),
  "elapsed_sec": round(time.time() - t0, 1),
  "environment": {"python": sys.version.split()[0], "platform": platform.platform(),
                  "packages": {m: __import__(m).__version__ for m in ("tensorflow", "numpy", "cv2")}},
  "fall_detection_eval": EVAL,
  "contents": [{"path": f.name, "bytes": f.stat().st_size, "sha256": _sha256(f)}
               for f in sorted(RUN_DIR.iterdir()) if f.is_file()],
  "note": "Evaluation logic verbatim from SHine-K_reproducibility_colab.ipynb; deployed thresholds (sens=1.0), no re-tuning.",
}
(RUN_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
zip_path = shutil.make_archive(str(BASE_OUTPUT / RUN_ID), "zip", root_dir=str(RUN_DIR))
print("zipped:", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    pass
print("DONE —", RUN_ID)